# Notebook 07 — Attentive CutMix

**Session:** Image Augmentation  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Implement **Attentive CutMix** (Walawalkar et al., 2020): pass an image through a convolutional backbone (ResNet-50 prior to global average pooling, yielding a $7 \times 7$ spatial feature grid on $224 \times 224$ inputs), identify the top-$k = 6$ highest-activation grid cells, and paste those semantic $32 \times 32$ patches onto a randomly paired target image.

**Learning objectives:**
- Extract spatial activation grids from a truncated ResNet-50 backbone.
- Select top-$k$ attentive patches using `np.argpartition` and `np.unravel_index`.
- Generate and visualize mixed images locally without Colab dependencies.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms.functional as tf
from PIL import Image

## 1  Attentive Region Selection & Patch Replacement

In [ ]:
def top_k_coords(activation_grid: np.ndarray, k: int = 6) -> np.ndarray:
    """Return (row, col) indices of the k highest-activation cells in a 2D grid."""
    flat_indices = np.argpartition(activation_grid.ravel(), activation_grid.size - k)[-k:]
    return np.column_stack(np.unravel_index(flat_indices, activation_grid.shape))


def get_attentive_regions(
    image: Image.Image,
    feature_extractor: nn.Module,
    device: torch.device,
    k: int = 6,
) -> np.ndarray:
    """Extract the top-k attentive grid coordinates from the last convolutional feature map."""
    x = tf.to_tensor(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = feature_extractor(x)
    last_feature_map = output[0].mean(dim=0).detach().cpu().numpy()
    return top_k_coords(last_feature_map, k=k)


def replace_attentive_region(
    np_target: np.ndarray,
    np_source: np.ndarray,
    region_coord: np.ndarray,
    grid_size: int = 32,
) -> None:
    """Copy a single grid_size x grid_size patch from np_source into np_target in place."""
    row, col = int(region_coord[0]), int(region_coord[1])
    y1, y2 = grid_size * row, grid_size * (row + 1)
    x1, x2 = grid_size * col, grid_size * (col + 1)
    np_target[y1:y2, x1:x2] = np_source[y1:y2, x1:x2]


def replace_attentive_regions(
    target_img: Image.Image,
    source_img: Image.Image,
    attentive_regions: np.ndarray,
    grid_size: int = 32,
) -> Image.Image:
    """Paste all attentive patches from source_img onto target_img."""
    np_target = np.array(target_img).copy()
    np_source = np.array(source_img)
    for region in attentive_regions:
        replace_attentive_region(np_target, np_source, region, grid_size=grid_size)
    return Image.fromarray(np_target)

## 2  Generate Attentive CutMix Samples Locally

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
except Exception:
    backbone = models.resnet50(weights=None)

feature_extractor = nn.Sequential(*list(backbone.children())[:-2]).to(device).eval()

source_dir = Path("../data/sample_dataset/test/cats")
output_dir = Path("outputs/attentive_cutmix")
output_dir.mkdir(parents=True, exist_ok=True)

img_paths = sorted(p for p in source_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
rng = np.random.default_rng(42)
grid_size = 32

mixed_examples: list[tuple[Image.Image, Image.Image, Image.Image]] = []
for i, img_path in enumerate(img_paths):
    src_img = Image.open(img_path).convert("RGB").resize((224, 224))
    candidates = [j for j in range(len(img_paths)) if j != i]
    rand_idx = int(rng.choice(candidates)) if candidates else i
    tgt_raw = Image.open(img_paths[rand_idx]).convert("RGB")
    ori_size = tgt_raw.size
    tgt_img = tgt_raw.resize((224, 224))

    regions = get_attentive_regions(src_img, feature_extractor, device, k=6)
    mixed_img = replace_attentive_regions(tgt_img, src_img, regions, grid_size=grid_size)
    mixed_img.resize(ori_size).save(output_dir / img_path.name)
    mixed_examples.append((src_img, tgt_img, mixed_img))

print(f"Saved {len(mixed_examples)} Attentive CutMix images to {output_dir}")

if mixed_examples:
    src_img, tgt_img, mixed_img = mixed_examples[0]
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
    for ax, im, title in zip(
        axes,
        [src_img, tgt_img, mixed_img],
        ["Source (Attentive)", "Target", "Attentive CutMix"],
        strict=True,
    ):
        ax.imshow(im)
        ax.set_title(title)
        ax.axis("off")
    fig.tight_layout()
    plt.show()